# 01 — XDF → FIF Conversion

Converts raw Lab Recorder XDF files to MNE-compatible FIF format.
One FIF file is produced per subject, along with an events file and an event mapping log.

**Input:** `<raw_dir>/<subject>.xdf`  
**Output:** `<analysis_dir>/<subject>/<subject>_raw.fif`, `_events_eve.fif`, `_event_mapping.txt`

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_raw_files, convert_xdf_to_fif

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

xdf_files = find_raw_files(cfg, extension='xdf')
print(f"Found {len(xdf_files)} XDF files")

In [ ]:
# ── Test conversion with the first subject ──
test_xdf = xdf_files[0]
print(f"Test conversion with: {test_xdf.name}\n")

success = convert_xdf_to_fif(cfg, test_xdf, overwrite=False, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")

In [ ]:
# ── Inspect the converted subject ──
import mne
import numpy as np
from eeg_toolkit import get_subject_dir, get_subject_path, find_subjects

subjects = find_subjects(cfg)
test_subject = subjects[0]

raw = mne.io.read_raw_fif(
    get_subject_path(cfg, test_subject, 'raw'),
    preload=False, verbose='WARNING'
)
events = mne.read_events(get_subject_path(cfg, test_subject, 'events'))

print(f"Raw: {raw}")
print(f"\nFirst 5 channels: {raw.ch_names[:5]}")
print(f"Sample rate: {raw.info['sfreq']} Hz")
print(f"\nNumber of events: {len(events)}")
print(f"First 5 events:\n{events[:5]}")

unique_codes, counts = np.unique(events[:, 2], return_counts=True)
print(f"\nUnique event codes ({len(unique_codes)}):")
for code, n in zip(unique_codes, counts):
    print(f"  {code}: {n}")

In [ ]:
# ── Verify EEG amplitude units (should be µV range after 1 Hz HP filter) ──
raw_filtered = raw.copy().load_data().filter(l_freq=1.0, h_freq=None, verbose='WARNING')
data = raw_filtered.get_data()
print(f"After 1 Hz high-pass filter:")
print(f"  min = {data.min()*1e6:.1f} µV")
print(f"  max = {data.max()*1e6:.1f} µV")
print(f"  std = {data.std()*1e6:.1f} µV")
print("\nExpected range: roughly ±100 µV std. Very large values suggest a unit scaling issue.")

In [ ]:
# ── Convert all subjects ──
from eeg_toolkit import convert_all_xdfs

summary = convert_all_xdfs(cfg, overwrite=False, verbose=True)

In [ ]:
# ── Verify all subjects converted ──
from eeg_toolkit import find_subjects, find_raw_files

all_folders = find_subjects(cfg, apply_exclusions=False)
print(f"All converted folders ({len(all_folders)}): {all_folders}")

included = find_subjects(cfg)
print(f"\nSubjects to analyze ({len(included)}): {included}")

xdf_subjects = sorted([f.stem for f in find_raw_files(cfg, extension='xdf')])
missing = set(xdf_subjects) - set(all_folders)
print(f"\nXDFs without a folder ({len(missing)}): {sorted(missing) if missing else 'none'}")